# DATASCI 451 - Bayesian Hierarchical Model Fitting

Fit and compare three models:
1. Complete Pooling
2. No Pooling
3. Partial Pooling (Hierarchical)

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import arviz as az
import warnings
warnings.filterwarnings('ignore')

# Choose backend: PyMC or Stan
USE_PYMC = True  # Set to False to use Stan

if USE_PYMC:
    import pymc as pm
    print(f"Using PyMC version: {pm.__version__}")
else:
    import cmdstanpy
    print(f"Using CmdStanPy")

print(f"ArviZ version: {az.__version__}")

# Plot settings
plt.rcParams['figure.figsize'] = (12, 6)
az.style.use('arviz-darkgrid')

## 1. Load Data

In [ ]:
# Load monthly aggregated data
df = pd.read_csv('data/selected_stations_monthly.csv')
print(f"Data shape: {df.shape}")
df.head(10)

In [ ]:
# Prepare data for modeling
# Create station and month indices (0-based for PyMC)
station_names = df['short_name'].unique()
station_idx = {name: i for i, name in enumerate(station_names)}
df['station_id'] = df['short_name'].map(station_idx)
df['month_id'] = df['Month'] - 1  # 0-indexed

# Extract arrays
y = df['TAVG_mean'].values
station = df['station_id'].values
month = df['month_id'].values

n_stations = len(station_names)
n_months = 4
n_obs = len(y)

print(f"Observations: {n_obs}")
print(f"Stations: {n_stations}")
print(f"Months: {n_months}")
print(f"\nStation names: {list(station_names)}")
print(f"\nY range: {y.min():.1f} to {y.max():.1f} °F")
print(f"Y mean: {y.mean():.1f} °F")

## 2. Model 1: Complete Pooling

$$y_{ij} \sim N(\mu + \beta_{m(j)}, \sigma^2)$$

All stations share the same baseline $\mu$.

In [ ]:
if USE_PYMC:
    with pm.Model() as model_complete_pooling:
        # Priors
        mu = pm.Normal('mu', mu=25, sigma=20)  # overall mean
        beta = pm.Normal('beta', mu=0, sigma=15, shape=n_months)  # month effects
        sigma = pm.HalfCauchy('sigma', beta=10)  # observation noise
        
        # Expected value
        mu_y = mu + beta[month]
        
        # Likelihood
        y_obs = pm.Normal('y_obs', mu=mu_y, sigma=sigma, observed=y)
        
        # Sample
        trace_cp = pm.sample(2000, tune=1000, cores=2, random_seed=451,
                             return_inferencedata=True)

print("Complete Pooling model fitted!")

In [ ]:
# Summary
az.summary(trace_cp, var_names=['mu', 'beta', 'sigma'])

## 3. Model 2: No Pooling

$$y_{ij} \sim N(\alpha_i + \beta_{m(j)}, \sigma^2)$$

Each station has its own independent intercept $\alpha_i$.

In [ ]:
if USE_PYMC:
    with pm.Model() as model_no_pooling:
        # Priors - independent station effects
        alpha = pm.Normal('alpha', mu=25, sigma=20, shape=n_stations)  # station intercepts
        beta = pm.Normal('beta', mu=0, sigma=15, shape=n_months)  # month effects
        sigma = pm.HalfCauchy('sigma', beta=10)
        
        # Expected value
        mu_y = alpha[station] + beta[month]
        
        # Likelihood
        y_obs = pm.Normal('y_obs', mu=mu_y, sigma=sigma, observed=y)
        
        # Sample
        trace_np = pm.sample(2000, tune=1000, cores=2, random_seed=451,
                             return_inferencedata=True)

print("No Pooling model fitted!")

In [ ]:
# Summary
az.summary(trace_np, var_names=['alpha', 'beta', 'sigma'])

## 4. Model 3: Partial Pooling (Hierarchical)

$$y_{ij} \sim N(\alpha_i + \beta_{m(j)}, \sigma^2)$$
$$\alpha_i \sim N(\mu_\alpha, \tau^2)$$

Station effects come from a common distribution — **the key hierarchical structure**.

In [ ]:
if USE_PYMC:
    with pm.Model() as model_hierarchical:
        # Hyperpriors for station effects
        mu_alpha = pm.Normal('mu_alpha', mu=25, sigma=20)  # population mean
        tau = pm.HalfCauchy('tau', beta=10)  # between-station SD
        
        # Station effects (non-centered parameterization for better sampling)
        alpha_offset = pm.Normal('alpha_offset', mu=0, sigma=1, shape=n_stations)
        alpha = pm.Deterministic('alpha', mu_alpha + tau * alpha_offset)
        
        # Month effects
        beta = pm.Normal('beta', mu=0, sigma=15, shape=n_months)
        
        # Observation noise
        sigma = pm.HalfCauchy('sigma', beta=10)
        
        # Expected value
        mu_y = alpha[station] + beta[month]
        
        # Likelihood
        y_obs = pm.Normal('y_obs', mu=mu_y, sigma=sigma, observed=y)
        
        # Sample
        trace_hier = pm.sample(2000, tune=1000, cores=2, random_seed=451,
                               return_inferencedata=True)

print("Hierarchical model fitted!")

In [ ]:
# Summary
az.summary(trace_hier, var_names=['mu_alpha', 'tau', 'alpha', 'beta', 'sigma'])

## 5. Convergence Diagnostics

In [ ]:
# Trace plots for hierarchical model
az.plot_trace(trace_hier, var_names=['mu_alpha', 'tau', 'sigma'])
plt.suptitle('Hierarchical Model - Trace Plots (Hyperparameters)', y=1.02)
plt.tight_layout()
plt.savefig('plots/10_trace_hyperparameters.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# R-hat diagnostics
rhat = az.rhat(trace_hier)
print("R-hat values (should be < 1.01):")
for var in ['mu_alpha', 'tau', 'sigma']:
    print(f"  {var}: {float(rhat[var].values):.4f}")

In [ ]:
# Effective sample size
ess = az.ess(trace_hier)
print("Effective Sample Size (should be > 400):")
for var in ['mu_alpha', 'tau', 'sigma']:
    print(f"  {var}: {float(ess[var].values):.0f}")

## 6. Model Comparison (WAIC / LOO)

In [ ]:
# Add log likelihood for model comparison
if USE_PYMC:
    with model_complete_pooling:
        pm.compute_log_likelihood(trace_cp)
    with model_no_pooling:
        pm.compute_log_likelihood(trace_np)
    with model_hierarchical:
        pm.compute_log_likelihood(trace_hier)

In [ ]:
# Compare models using WAIC
model_comparison = az.compare(
    {
        'Complete Pooling': trace_cp,
        'No Pooling': trace_np,
        'Hierarchical': trace_hier
    },
    ic='waic'
)

print("Model Comparison (WAIC - lower is better):")
model_comparison

In [ ]:
# Visualize comparison
az.plot_compare(model_comparison, insample_dev=False)
plt.title('Model Comparison (WAIC)')
plt.tight_layout()
plt.savefig('plots/11_model_comparison_waic.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Save Results

In [ ]:
# Save traces
trace_cp.to_netcdf('data/trace_complete_pooling.nc')
trace_np.to_netcdf('data/trace_no_pooling.nc')
trace_hier.to_netcdf('data/trace_hierarchical.nc')

# Save station names mapping
pd.DataFrame({'station_name': station_names, 'station_id': range(n_stations)}).to_csv(
    'data/station_mapping.csv', index=False)

print("Results saved!")

---
## Summary

Three models fitted:
1. **Complete Pooling**: Single baseline for all stations
2. **No Pooling**: Independent baselines per station
3. **Hierarchical**: Station baselines from common distribution

Next: Run `04_results_visualization.ipynb` for detailed analysis.